In [2]:
import pandas as pd
from Bio import Entrez, SeqIO
import time

In [7]:
def extrae_gene_symbols(dataframe):
    
    df_genes = dataframe["gene_symbol"]
    
    df_genes = df_genes.dropna()
    
    df_genes = df_genes.reset_index(drop=False)["gene_symbol"]
    
    lista_symbols = []
    
    for i, gene in enumerate(df_genes):
        
        gene = gene.replace(",", ";")
    
        if "dist" in gene:
            continue

        elif ";" in gene:

            separacion1 = gene.split(";")

            for gen in separacion1:
                lista_symbols.append(gen)

        else:
            lista_symbols.append(gene)
            
    return lista_symbols

In [ ]:
Entrez.email = "jbs1009@alu.ubu.es"

def descargar_secuencia(gene_symbol):    
    
    query = f"{gene_symbol}[Gene Name] AND Homo sapiens[Organism] AND refseq[filter] AND biomol_mrna[prop]"
    
    try:
        
        handle = Entrez.esearch(db = "nucleotide", term = query, retmax = 1)
        record = Entrez.read(handle)
        handle.close()
        
        if not record["IdList"]:
            return None
        
        seq_id = record["IdList"][0]
        
        fetch_handle = Entrez.efetch(db = "nucleotide", id = seq_id, rettype = "fasta", retmode = "text")
        
        seq_record = SeqIO.read(fetch_handle, "fasta")
        fetch_handle.close()
        
        return str(seq_record.seq)
    
    except Exception as e:
        
        print(f"Error al descargar {gene_symbol}: {e}")
        return None
        
    

In [4]:
#Esta versión aún no funciona
def limpia_gene_symbol(dataframe):
    
    df_genes = dataframe["gene_symbol"]
    
    df_genes = df_genes.dropna()
    
    for i, gene in enumerate(df_genes):
        
        if "dist" in gene:
            df_genes = df_genes.drop(i)
            df_genes = df_genes.reset_index(drop=True)
            
        if ";" in gene:
            df_genes = df_genes.drop(i)
            df_genes = df_genes.reset_index(drop=True)
            
            separacion = gene.split(";")
            
            nuevo_gen1 = pd.Series([separacion[0]], index = [len(df_genes)])
            
            df_genes = pd.concat([df_genes, nuevo_gen1])
            
            nuevo_gen2 = pd.Series([separacion[1]], index = [len(df_genes)])
            
            df_genes = pd.concat([df_genes, nuevo_gen2])
            
        if "," in gene:
            df_genes = df_genes.drop(i)
            df_genes = df_genes.reset_index(drop=True)
            
            separacion = gene.split(",")
            
            nuevo_gen1 = pd.Series([separacion[0]], index = [len(df_genes)])
            
            df_genes = pd.concat([df_genes, nuevo_gen1])
            
            nuevo_gen2 = pd.Series([separacion[1]], index = [len(df_genes)])
            
            df_genes = pd.concat([df_genes, nuevo_gen2])
            
    df_genes = df_genes.unique()

    return df_genes
            
            

In [5]:
df = pd.read_csv("download/t_common_variant.txt", sep = "\t", index_col = False)

In [8]:
lista_symbols = extrae_gene_symbols(df)

In [3]:
df.head()

,Chr,gene_symbol,SNPs_symbol,SNP_position,effect_allele,alternate_allele,joint_phase_P,joint_phase_OR,joint_phase_OR_CI,pubMed_ID,Unnamed: 10
0,6,GPR126,rs757765789,142758601,T,G,1.42E-06,1.06,1.016-1.098,28256260,NaN
1,1,SYT11,rs202015799,155839054,C,T,4.70E-09,-,-,24842889,NaN
2,12,SLC2A13,rs1994090,40428561,G,T,3.20E-54,12.05,8.35-17.41,24842889,NaN
3,12,SLC2A13,rs2708453,40478652,G,T,3.62E-54,12.05,8.35-17.41,24842889,NaN
4,12,SLC2A13,rs4768212,40474147,C,T,3.62E-54,12.05,8.35-17.41,24842889,NaN


In [54]:
df_genes = df["gene_symbol"]

In [55]:
df_genes

0        GPR126
1         SYT11
2       SLC2A13
3       SLC2A13
4       SLC2A13
         ...   
1053     INPP5F
1054       RIT2
1055       GCH1
1056    SIPA1L2
1057    TMPRSS9
Name: gene_symbol, Length: 1058, dtype: object

In [56]:
df_genes_t = df_genes.dropna()

In [57]:
df_genes_t = df_genes_t.reset_index(drop=False)["gene_symbol"]

In [58]:
df_genes_t

0        GPR126
1         SYT11
2       SLC2A13
3       SLC2A13
4       SLC2A13
         ...   
1051     INPP5F
1052       RIT2
1053       GCH1
1054    SIPA1L2
1055    TMPRSS9
Name: gene_symbol, Length: 1056, dtype: object

In [59]:
lista_symbols = []

for i, gene in enumerate(df_genes_t):
    
    if "dist" in gene:
        continue
    
    elif ";" in gene:
        
        separacion1 = gene.split(";")
        
        for gen in separacion1:
            lista_symbols.append(gen)
    
    elif "," in gene:
        
        separacion2 = gene.split(",")
        
        for gen in separacion2:
            lista_symbols.append(gen)
            
    else:
        lista_symbols.append(gene)

In [65]:
len(set(lista_symbols))

663

In [48]:
df_genes_t.head(20)

0              GPR126
1               SYT11
2             SLC2A13
3             SLC2A13
4             SLC2A13
5             SLC2A15
6               LRRK2
7               LRRK2
8         GPRIN3;SNCA
9     LINC02210-CRHR1
10        GPRIN3;SNCA
11    LINC02210-CRHR1
12           SERPINA1
13        S1PR1;OLFM3
14               MAPT
15               MAPT
16    LINC02210-CRHR1
17    LINC02210-CRHR1
18            HLA-DRA
19               DGKQ
dtype: object

In [266]:
for i in df_genes_t:
    print(i)

GPR126
SYT11
SLC2A13
SLC2A13
SLC2A13
SLC2A15
LRRK2
LRRK2
GPRIN3;SNCA
LINC02210-CRHR1
GPRIN3;SNCA
LINC02210-CRHR1
SERPINA1
S1PR1;OLFM3
MAPT
MAPT
LINC02210-CRHR1
LINC02210-CRHR1
HLA-DRA
DGKQ
NSF
PRRG4;QSER1
GAK
WNT3
CSMD1
WNT3
UNC13B
LINC00693
MPHOSPH10
ZNF519
AAK1
DHRS2;DHRS4-AS1
C12orf75;CASC18
CDH6
GRB10
PIK3CD
LY75-CD302;PLA2R1
SH3GL2
PLA2R1
MED13;TBC1D3P2
CAST
LINC01815;DHFRP3
SPPL2C
DNAH11
PLA2R1
LINC01815;DHFRP3
CDH6
LINC01271;PTPN1
GABRB3
RNF130
NONE;FSCB
LINC01815;DHFRP3
MCTP2;LOC440311
LINC02210-CRHR1
LINC01815;DHFRP3
DPY19L3;PDCD5
HLA-DQB1;HLA-DQA2
SVOPL;ATP6V0A4
MIR548AG2;LINC01718
RIPK2
MAPT
SPPL2C
ANKRD36;ANKRD36
MED13;TBC1D3P2
POMGNT2;SNRK
43160
SLC2A14
SLC2A13
AP3B1
AP3B1
LAPTM4A;SDC1
TMTC1;IPO8
SND1;MIR129-1
LINC00693
ABCA3
HLX-AS1
LINC02471;LRRK2
FER
GPRIN3;SNCA
LOC100133091;DTX2P1-UPK3BP1-PMS2P11
PRKG1
CHN2
GAK
SLC25A48
MAPT
MAPT
TMEM175
MAPT
KANSL1
LINC02210
AP3B1
CASC6;EPHA7
CASC6;EPHA7
GPAT3;LOC101928978
GPAT3;LOC101928978
IL2RA
LINC01572
NDN;PWRN4
NDN;PWRN4
NEDD9
P

In [227]:
df_genes_t.head(10)

0         GPR126
1          SYT11
2        SLC2A13
3        SLC2A13
4        SLC2A13
5        SLC2A15
6          LRRK2
7          LRRK2
8    GPRIN3;SNCA
9           MAPT
dtype: object

In [198]:
a = df_genes_t.drop(6)

In [199]:
len(a)

1055

In [200]:
b = a.reset_index(drop = False)["gene_symbol"]

In [203]:
len(b)

1055

In [124]:
for i in df_genes_t:
    print(i)

GPR126
SYT11
SLC2A13
SLC2A13
SLC2A13
SLC2A15
LRRK2
LRRK2
GPRIN3;SNCA
LINC02210-CRHR1
GPRIN3;SNCA
LINC02210-CRHR1
SERPINA1
S1PR1;OLFM3
MAPT
MAPT
LINC02210-CRHR1
LINC02210-CRHR1
HLA-DRA
DGKQ
NSF
PRRG4;QSER1
GAK
WNT3
CSMD1
WNT3
UNC13B
LINC00693
MPHOSPH10
ZNF519
AAK1
DHRS2;DHRS4-AS1
C12orf75;CASC18
CDH6
GRB10
PIK3CD
LY75-CD302;PLA2R1
SH3GL2
PLA2R1
MED13;TBC1D3P2
CAST
LINC01815;DHFRP3
SPPL2C
DNAH11
PLA2R1
LINC01815;DHFRP3
CDH6
LINC01271;PTPN1
GABRB3
RNF130
NONE;FSCB
LINC01815;DHFRP3
MCTP2;LOC440311
LINC02210-CRHR1
LINC01815;DHFRP3
DPY19L3;PDCD5
HLA-DQB1;HLA-DQA2
SVOPL;ATP6V0A4
MIR548AG2;LINC01718
RIPK2
MAPT
SPPL2C
ANKRD36;ANKRD36
MED13;TBC1D3P2
POMGNT2;SNRK
43160
SLC2A14
SLC2A13
AP3B1
AP3B1
LAPTM4A;SDC1
TMTC1;IPO8
SND1;MIR129-1
LINC00693
ABCA3
HLX-AS1
LINC02471;LRRK2
FER
GPRIN3;SNCA
LOC100133091;DTX2P1-UPK3BP1-PMS2P11
PRKG1
CHN2
GAK
SLC25A48
MAPT
MAPT
TMEM175
MAPT
KANSL1
LINC02210
AP3B1
CASC6;EPHA7
CASC6;EPHA7
GPAT3;LOC101928978
GPAT3;LOC101928978
IL2RA
LINC01572
NDN;PWRN4
NDN;PWRN4
NEDD9
P

In [44]:
a = df_genes_t[6]

In [45]:
a

'LINC02471;LRRK2'

In [63]:
df_genes_t

0        GPR126
1         SYT11
2       SLC2A13
3       SLC2A13
4       SLC2A13
         ...   
1053     INPP5F
1054       RIT2
1055       GCH1
1056    SIPA1L2
1057    TMPRSS9
Name: gene_symbol, Length: 1056, dtype: object

In [71]:
df_genes_t = df_genes_t.reset_index(drop=True)

In [72]:
df_genes_t

0        GPR126
1         SYT11
2       SLC2A13
3       SLC2A13
4       SLC2A13
         ...   
1051     INPP5F
1052       RIT2
1053       GCH1
1054    SIPA1L2
1055    TMPRSS9
Name: gene_symbol, Length: 1056, dtype: object

In [76]:
a.split(";")[0]

'LINC02471'

In [94]:
nueva_instancia = pd.Series([a.split(";")[0]], index = [len(df_genes_t)])

In [95]:
nueva_instancia

1056    LINC02471
dtype: object

In [96]:
serie_final = pd.concat([df_genes_t, nueva_instancia])

In [97]:
serie_final

0          GPR126
1           SYT11
2         SLC2A13
3         SLC2A13
4         SLC2A13
          ...    
1052         RIT2
1053         GCH1
1054      SIPA1L2
1055      TMPRSS9
1056    LINC02471
Length: 1057, dtype: object

In [11]:
secuencias_guardadas = {}

for gen in lista_symbols[:5]:
    print(f"Buscando: {gen}", end = " ")
    
    secuencia = descargar_secuencia(gen)
    
    if secuencia:
        secuencias_guardadas[gen] = secuencia
        print("Encontrado.")
    else:
        print("No encontrado.")
    
    time.sleep(0.5)

Buscando: GPR126 Encontrado
Buscando: SYT11 Encontrado
Buscando: SLC2A13 Encontrado
Buscando: SLC2A13 Encontrado
Buscando: SLC2A13 Encontrado
Prueba terminada


In [14]:
len(secuencias_guardadas["GPR126"])

6280